# Scratch-coder Stage B aggregate review

In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT/'.git').exists(): ROOT=ROOT.parent
if not (ROOT/'.git').exists(): raise RuntimeError('Repository root not found')
sys.path.insert(0,str(ROOT))
import json,pandas as pd,numpy as np
from analysis.scratch_coder_stage_a import validate_export
from analysis.scratch_coder_stage_b import build_stage_b_data,run_stage_b_analysis
from analysis.scratch_coder_stage_b.metrics import prf,tagstats,kappa
from analysis.validation.bootstrap import percentile
OUT=ROOT/'analysis/outputs_validation_scratch_stage_b_20260825'

## 1. Frozen authorities / hashes
## 2. Panel reconciliation
## 3. Label support
## 4. Exact-set agreement
## 5. Jaccard
## 6. Per-label contingencies
## 7. Per-label pairwise kappa
## 8. Model precision/recall/F1
## 9. Macro performance
## 10. Tag diagnostics

In [ ]:
meta=json.loads((OUT/'run_metadata.json').read_text()); data=build_stage_b_data(); assert (len(data.responses),len(data.baseline_ids),len(data.hard_case_ids))==(675,150,75)
support=pd.read_csv(OUT/'label_support.csv'); exact=pd.read_csv(OUT/'exact_set_jaccard_summary.csv'); cont=pd.read_csv(OUT/'per_label_contingencies.csv'); kap=pd.read_csv(OUT/'per_label_pairwise_kappa.csv'); perf=pd.read_csv(OUT/'per_label_model_performance.csv'); macro=pd.read_csv(OUT/'macro_performance.csv'); tags=pd.read_csv(OUT/'tag_diagnostics.csv')
(meta['verified_authorities'],support,exact,cont,kap,perf,macro,tags)

## 11. Bootstrap checks
## 12. Assertions against saved output CSVs

In [ ]:
r=perf.iloc[0]; c=cont[(cont['dimension']==r['dimension'])&(cont['label']==r['label'])].iloc[0]; z=prf({'tp':int(c.tp),'fp':int(c.fp),'fn':int(c.fn),'tn':int(c.tn)}); assert all(np.isclose(z[k],r[k]) for k in z)
t=tags.iloc[0]; ref=[1]*int(t.tp)+[0]*int(t.fp)+[1]*int(t.fn)+[0]*int(t.tn); pred=[1]*int(t.tp)+[1]*int(t.fp)+[0]*int(t.fn)+[0]*int(t.tn); assert np.isclose(tagstats(ref,pred)['gwet_ac1'],t.gwet_ac1)
b=pd.read_csv(OUT/'bootstrap_per_label_performance.csv');v=b[(b['dimension']==r['dimension'])&(b['label']==r['label'])].precision.dropna().tolist();assert np.isclose(percentile(v,.025),r.precision_ci_lower);assert np.isclose(percentile(v,.975),r.precision_ci_upper)
assert len(exact)==12 and set(kap['pair'])=={'A-B','A-C','B-C','L-A','L-B','L-C'}